In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path.cwd().parent

UTA_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "sequences"
)

SAVE_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "embeddings"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "checkpoints"
    / "best_cnn_3class.pth"
)

SAVE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
model = models.efficientnet_b0(
    weights=None
)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 3)
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)
model.eval()

In [ ]:
feature_extractor = nn.Sequential(
    model.features,
    model.avgpool,
    nn.Flatten(),
    model.classifier[0],
    model.classifier[1],
    model.classifier[2]
).to(device)

feature_extractor.eval()

In [ ]:
transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

In [ ]:
x = torch.randn(
    1,
    3,
    224,
    224
).to(device)

with torch.no_grad():
    emb = feature_extractor(x)

print(emb.shape)

In [ ]:
folder_to_name = {
    "0": "alert",
    "5": "low_vigilant",
    "10": "drowsy"
}

In [ ]:
subjects = sorted(
    [p.name for p in UTA_ROOT.iterdir()]
)

print(subjects)
print(len(subjects))

In [ ]:
for subject in subjects:

    print(f"\nProcessing subject {subject}")

    subject_dir = UTA_ROOT / subject
    save_dir = SAVE_ROOT / subject
    save_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for cls in ["0", "5", "10"]:

        frame_dir = subject_dir / cls

        frame_paths = sorted(
            frame_dir.glob("*.jpg")
        )

        embeddings = []

        for img_path in tqdm(
            frame_paths,
            desc=f"{subject}-{cls}"
        ):

            img = (
                Image.open(img_path)
                .convert("RGB")
            )

            x = (
                transform(img)
                .unsqueeze(0)
                .to(device)
            )

            with torch.no_grad():
                emb = feature_extractor(x)

            emb = (
                emb.squeeze()
                .cpu()
                .numpy()
            )

            embeddings.append(emb)

        embeddings = np.array(
            embeddings,
            dtype=np.float32
        )

        save_path = (
            save_dir
            / f"{folder_to_name[cls]}.npy"
        )

        np.save(
            save_path,
            embeddings
        )

        print(
            f"Saved {save_path.name}:",
            embeddings.shape
        )

In [ ]:
for subject in subjects[:3]:

    print(f"\nSubject {subject}")

    for name in [
        "alert",
        "low_vigilant",
        "drowsy"
    ]:

        arr = np.load(
            SAVE_ROOT
            / subject
            / f"{name}.npy"
        )

        print(
            name,
            arr.shape
        )

In [ ]:
import shutil
from pathlib import Path

base = PROJECT_ROOT / "processed" / "UTA"

# Remove bad sequences
shutil.rmtree(base / "sequences" / "46", ignore_errors=True)

# Remove bad embeddings
shutil.rmtree(base / "embeddings" / "46", ignore_errors=True)

# Remove any partially created normalized embeddings
shutil.rmtree(base / "normalized_embeddings" / "46", ignore_errors=True)

# Remove any partially created windows
shutil.rmtree(base / "windows" / "46", ignore_errors=True)

print("✅ Deleted all processed data for subject 46.")